# Cobalt L-edge energy sweep: multi-energy phase retrieval

Load the same cobalt energy-sweep HDF5 chunks used by the universal notebook, apply the same detector-energy normalization and support preparation, then keep one circular polarization so the stack can be passed to `phase_retrieval_core_multienergy.py`.

In [ ]:
from pathlib import Path
import json
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

# Make imports work when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "library").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from library import phase_retrieval_core_multienergy as pr
from library import interactive


plt.rcParams.update({"figure.figsize": (7, 5), "image.cmap": "gray"})


## Load the cobalt energy chunks

Each chunk contains ideal CR and CL holograms, a beamstop mask, a support mask, and effective refractive-index arrays. Both polarizations are loaded for inspection, but only `SELECTED_POLARIZATION` is passed to the multi-energy reconstruction.

In [ ]:
DATA_DIR = REPO_ROOT / "notebooks" / "prop_False_proj_False"
MANIFEST_FILE = DATA_DIR / "manifest.json"
load_hologram = "ideal_holograms"
SELECTED_POLARIZATION = "CR"  # "CR" or "CL"

if SELECTED_POLARIZATION not in {"CR", "CL"}:
    raise ValueError("SELECTED_POLARIZATION must be 'CR' or 'CL'")
if not MANIFEST_FILE.exists():
    raise FileNotFoundError(MANIFEST_FILE)

with MANIFEST_FILE.open("r") as handle:
    manifest = json.load(handle)[::2]

chunk_entries = sorted(manifest, key=lambda item: item["index"])
chunk_files = [DATA_DIR / entry["file"] for entry in chunk_entries]
missing_files = [path for path in chunk_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing energy chunk files: {missing_files}")

energies_eV = np.asarray([entry["energy_eV"] for entry in chunk_entries], dtype=float)
group_names = [path.stem for path in chunk_files]

cr_ideal = []
cl_ideal = []
beamstop_masks = []
supportmasks = []
effective_refractive_indices_m = []
for chunk_index, (expected_energy, chunk_file) in enumerate(zip(energies_eV, chunk_files)):
    with h5py.File(chunk_file, "r") as handle:
        file_energy = float(handle.attrs["energy_eV"])
        if not np.isclose(file_energy, expected_energy):
            raise ValueError(
                f"Manifest energy {expected_energy} eV does not match "
                f"{chunk_file.name} energy {file_energy} eV"
            )

        cr_ideal.append(np.squeeze(np.asarray(handle[f"{load_hologram}/CR"], dtype=float)))
        cl_ideal.append(np.squeeze(np.asarray(handle[f"{load_hologram}/CL"], dtype=float)))
        beamstop_masks.append(np.asarray(handle["beamstop_mask/data"], dtype=float))
        supportmasks.append(np.asarray(handle["supportmask/data"], dtype=float))
        effective_refractive_indices_m.append(
            np.asarray(handle["material_layers/effective_refractive_indices_m"])
        )

        if chunk_index == 0:
            layer_names = np.asarray(handle["material_layers/layer_names"]).astype(str)
            refractive_index_channel_names = np.asarray(
                handle["material_layers/refractive_index_channel_names"]
            ).astype(str)

cr_ideal = np.stack(cr_ideal)
cl_ideal = np.stack(cl_ideal)
beamstop_masks = np.stack(beamstop_masks)
supportmasks = np.stack(supportmasks)
effective_refractive_indices_m = np.stack(effective_refractive_indices_m)
emin_index = int(np.argmin(energies_eV))
emin_group = group_names[emin_index]
supportmask = supportmasks[emin_index]

if cr_ideal.shape != cl_ideal.shape:
    raise ValueError(f"CR and CL shapes differ: {cr_ideal.shape} vs {cl_ideal.shape}")
if cr_ideal.shape[0] != len(energies_eV):
    raise ValueError("The hologram energy axis does not match energies")
if np.any(~np.isfinite(cr_ideal)) or np.any(~np.isfinite(cl_ideal)):
    raise ValueError("The ideal holograms contain NaN or infinite values")
if np.min(cr_ideal) < 0 or np.min(cl_ideal) < 0:
    raise ValueError("The ideal holograms must be non-negative intensities")

n_energy, ny, nx = cr_ideal.shape
print(f"energies: {energies_eV[0]:.1f} to {energies_eV[-1]:.1f} eV ({n_energy} points)")
print("CR stack:", cr_ideal.shape, cr_ideal.dtype)
print("CL stack:", cl_ideal.shape, cl_ideal.dtype)
print("selected polarization:", SELECTED_POLARIZATION)
print("effective refractive indices:", effective_refractive_indices_m.shape, effective_refractive_indices_m.dtype)
print("refractive-index channels:", list(refractive_index_channel_names))
print(f"Emin support: {emin_group} at {energies_eV[emin_index]:.1f} eV")


## Plot effective refractive indices

Select a layer by integer index or exact layer name to inspect the complex effective refractive-index channels stored in the HDF5 files.

In [ ]:
%matplotlib widget
plt.close("all")

REFRACTIVE_INDEX_LAYER = -1  # int index or layer name string

if isinstance(REFRACTIVE_INDEX_LAYER, str):
    matches = np.flatnonzero(layer_names == REFRACTIVE_INDEX_LAYER)
    if matches.size == 0:
        raise ValueError(f"Unknown layer name: {REFRACTIVE_INDEX_LAYER!r}")
    layer_index = int(matches[0])
else:
    layer_index = int(REFRACTIVE_INDEX_LAYER) % len(layer_names)

layer_n_eff = effective_refractive_indices_m[:, layer_index, :]
layer_label = layer_names[layer_index]

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for channel_index, channel_name in enumerate(refractive_index_channel_names):
    values = layer_n_eff[:, channel_index]
    axes[0].plot(energies_eV, np.real(values-values[0]), "o-", label=channel_name)
    axes[1].plot(energies_eV, np.imag(values-values[0]), "o-", label=channel_name)
axes[0].set_ylabel("real(n_eff)")
axes[1].set_ylabel("imag(n_eff)")
axes[1].set_xlabel("Energy (eV)")
axes[0].set_title(f"Effective refractive indices: layer {layer_index} ({layer_label})")
for axis in axes:
    axis.grid(True, alpha=0.3)
    axis.legend()
plt.tight_layout()


## Inspect raw holograms

In [ ]:
%matplotlib widget
plt.close("all")

show_indices = np.unique(np.linspace(0, n_energy - 1, 6, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices) * 3, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log10(np.abs(np.fft.fftshift(np.fft.fft2(stack[index])))))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram Fourier magnitudes, log display")
plt.tight_layout()


## Normalize detector sampling

This is the same detector-space preparation used in `10_cobalt_energy_sweep_universal.ipynb`: each hologram is rescaled by `E/Emin`, then the optional pre-retrieval crop/bin controls are applied. The support template is prepared with the same support-specific rules.

In [ ]:
rescale=False
# Optional phase-retrieval preprocessing controls.
# Examples: PRE_RETRIEVAL_CROP_SHAPE = (384, 384), PRE_RETRIEVAL_BIN_FACTOR = 2.
PRE_RETRIEVAL_CROP_SHAPE = None
PRE_RETRIEVAL_BIN_FACTOR = 1
USE_MASK_PIXEL = load_hologram != "ideal_holograms"

def centered_rescale(image, scale, output_shape=None, order=1, rescale=True):
    """Rescale about the array center and return a fixed-size center crop."""
    image = np.asarray(image)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if scale <= 0 or not np.isfinite(scale):
        raise ValueError("scale must be positive and finite")
    if output_shape is None:
        output_shape = image.shape
    output_shape = tuple(int(value) for value in output_shape)
    if len(output_shape) != 2 or min(output_shape) <= 0:
        raise ValueError("output_shape must contain two positive integers")
    if np.isclose(scale, 1.0) and output_shape == image.shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    output_indices = np.indices(output_shape, dtype=float)
    coordinates = (
        (output_indices.reshape(2, -1) - output_center[:, None]) / float(scale)
        + input_center[:, None]
    )
    if rescale:
        return ndi.map_coordinates(
            image,
            coordinates,
            order=order,
            mode="constant",
            cval=0.0,
            prefilter=False,
        ).reshape(output_shape)
    else:
        return image

def centered_resize(image, output_shape, order=1):
    """Resize a 2D image to output_shape while preserving relative position."""
    image = np.asarray(image)
    output_shape = tuple(int(value) for value in output_shape)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if len(output_shape) != 2 or min(output_shape) <= 0:
        raise ValueError("output_shape must contain two positive integers")
    if image.shape == output_shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    matrix = np.diag(np.asarray(image.shape, dtype=float) / np.asarray(output_shape, dtype=float))
    offset = input_center - matrix @ output_center
    return ndi.affine_transform(
        image,
        matrix=matrix,
        offset=offset,
        output_shape=output_shape,
        order=order,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )

def normalized_crop_shape(crop_shape, image_shape):
    """Return a validated crop shape or None when cropping is disabled."""
    if crop_shape is None:
        return None
    if isinstance(crop_shape, (int, np.integer)):
        crop_shape = (int(crop_shape), int(crop_shape))
    crop_shape = tuple(int(value) for value in crop_shape)
    if len(crop_shape) != 2 or min(crop_shape) <= 0:
        raise ValueError("PRE_RETRIEVAL_CROP_SHAPE must be None, an int, or (ny, nx)")
    if crop_shape[0] > image_shape[0] or crop_shape[1] > image_shape[1]:
        raise ValueError("PRE_RETRIEVAL_CROP_SHAPE cannot exceed the hologram shape")
    return crop_shape

def center_crop_2d(image, output_shape):
    """Center-crop a 2D image to output_shape."""
    image = np.asarray(image)
    output_shape = tuple(int(value) for value in output_shape)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if output_shape[0] > image.shape[0] or output_shape[1] > image.shape[1]:
        raise ValueError("Cannot center-crop to a larger shape")
    start_y = (image.shape[0] - output_shape[0]) // 2
    start_x = (image.shape[1] - output_shape[1]) // 2
    return image[start_y:start_y + output_shape[0], start_x:start_x + output_shape[1]].copy()

def center_crop_stack(stack, output_shape):
    """Center-crop the last two axes of a stack."""
    stack = np.asarray(stack)
    output_shape = tuple(int(value) for value in output_shape)
    if output_shape[0] > stack.shape[-2] or output_shape[1] > stack.shape[-1]:
        raise ValueError("Cannot center-crop to a larger shape")
    start_y = (stack.shape[-2] - output_shape[0]) // 2
    start_x = (stack.shape[-1] - output_shape[1]) // 2
    return stack[..., start_y:start_y + output_shape[0], start_x:start_x + output_shape[1]].copy()

def bin_stack(stack, factor, reducer="mean"):
    """Bin the last two axes of a stack after cropping to a multiple of factor."""
    factor = int(factor)
    if factor <= 0:
        raise ValueError("PRE_RETRIEVAL_BIN_FACTOR must be a positive integer")
    stack = np.asarray(stack)
    if factor == 1:
        return stack.copy()

    binned_shape = (stack.shape[-2] // factor, stack.shape[-1] // factor)
    if min(binned_shape) <= 0:
        raise ValueError("PRE_RETRIEVAL_BIN_FACTOR is too large for the image shape")
    crop_shape = (binned_shape[0] * factor, binned_shape[1] * factor)
    cropped = center_crop_stack(stack, crop_shape)
    reshaped = cropped.reshape(*cropped.shape[:-2], binned_shape[0], factor, binned_shape[1], factor)
    if reducer == "mean":
        return reshaped.mean(axis=(-3, -1))
    if reducer == "max":
        return reshaped.max(axis=(-3, -1))
    raise ValueError("reducer must be 'mean' or 'max'")

def build_support_template(support):
    """Build the shifted support template used for phase retrieval."""
    from skimage.draw import disk

    support = np.asarray(support) != 0
    structure = np.zeros((15, 15), dtype=bool)
    yy, xx = disk((structure.shape[0] // 2, structure.shape[1] // 2), 5)
    structure[yy, xx] = True

    shift_x = -(167 - support.shape[1] // 2)
    shift_y = -(345 - support.shape[0] // 2)
    rolled_dilated = np.roll(
        np.roll(ndi.binary_dilation(support, structure=structure), shift=shift_x, axis=1),
        shift=shift_y,
        axis=0,
    )
    rolled_support = np.roll(
        np.roll(support, shift=shift_x, axis=1),
        shift=shift_y,
        axis=0,
    )
    template = rolled_dilated.astype(bool)
    x_cut = min(300, template.shape[1])
    rolled_support=ndi.binary_erosion(rolled_support,iterations=3)
    template[:, :x_cut] = rolled_support[:, :x_cut]

    template=np.roll(np.roll(template, shift=-shift_x, axis=1), shift=-shift_y, axis=0)
   
    return template

def preprocess_detector_stack(stack, crop_shape, bin_factor, reducer="mean"):
    """Apply the detector-space crop/bin operations to holograms or masks."""
    prepared = np.asarray(stack).copy()
    if crop_shape is not None:
        prepared = center_crop_stack(prepared, crop_shape)
    if bin_factor > 1:
        prepared = bin_stack(prepared, bin_factor, reducer=reducer)
    return prepared

def preprocess_support_template(template, crop_shape, bin_factor, final_shape):
    """Apply the support-specific crop/bin rules requested for this notebook."""
    prepared = np.asarray(template) != 0
    if crop_shape is not None:
        # Cropping detector data changes the array shape. For the support, keep
        # the same fractional position and size instead of cutting pixels away.
        prepared = centered_resize(prepared.astype(float), crop_shape, order=0) > 0.5
    if bin_factor > 1:
        # Detector binning reduces the Fourier grid. For the support template,
        # keep the central real-space region with the final binned shape.
        prepared = center_crop_2d(prepared, final_shape) != 0
    return prepared.astype(bool)



emin_eV = float(energies_eV[emin_index])
scale_factors = energies_eV / emin_eV
cr_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=5, rescale=rescale)
    for image, scale in zip(cr_ideal, scale_factors)
])
cl_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=5, rescale=rescale)
    for image, scale in zip(cl_ideal, scale_factors)
])
mask_pixel_energy = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=5, rescale=rescale)
    for image, scale in zip(beamstop_masks, scale_factors)
])

# Linear interpolation of non-negative inputs should remain non-negative;
# clip tiny floating-point undershoots defensively.
cr_rescaled = np.maximum(cr_rescaled, 0.0)
cl_rescaled = np.maximum(cl_rescaled, 0.0)
mask_pixel_energy = np.maximum(mask_pixel_energy, 0.0)

crop_shape = normalized_crop_shape(PRE_RETRIEVAL_CROP_SHAPE, (ny, nx))
bin_factor = int(PRE_RETRIEVAL_BIN_FACTOR)
supptemp_full = build_support_template(supportmask)

cr_prepared = preprocess_detector_stack(cr_rescaled, crop_shape, bin_factor, reducer="mean")
cl_prepared = preprocess_detector_stack(cl_rescaled, crop_shape, bin_factor, reducer="mean")
mask_pixel_energy = preprocess_detector_stack(mask_pixel_energy, crop_shape, bin_factor, reducer="max")
supptemp = preprocess_support_template(
    supptemp_full,
    crop_shape,
    bin_factor,
    final_shape=cr_prepared.shape[-2:],
)
ny_pr, nx_pr = cr_prepared.shape[-2:]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(supportmask)
axes[0].set_title(f"Raw support: {emin_eV:.1f} eV")
axes[1].imshow(supptemp)
axes[1].set_title("Support for retrieval")
axes[2].imshow(np.log1p(cr_ideal[-1]))
axes[2].set_title(f"Raw CR at {energies_eV[-1]:.1f} eV")
axes[3].imshow(np.log1p(cr_prepared[-1]))
axes[3].set_title("Prepared CR for retrieval")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

print(f"Raw support pixels: {int(np.sum(supportmask != 0))} / {supportmask.size}")
print(f"Retrieval support pixels: {int(supptemp.sum())} / {supptemp.size}")
print(f"Scale-factor range: {scale_factors.min():.6f} to {scale_factors.max():.6f}")
print(f"Pre-retrieval crop shape: {crop_shape}")
print(f"Pre-retrieval bin factor: {bin_factor}")
print("Prepared stacks:", cr_prepared.shape, cl_prepared.shape)
print("Prepared mask_pixel:", mask_pixel_energy.shape, "use mask:", USE_MASK_PIXEL)

## Build the single-polarization multi-energy stack

In [ ]:
prepared_by_polarization = {
    "CR": cr_prepared,
    "CL": cl_prepared,
}

holograms = prepared_by_polarization[SELECTED_POLARIZATION].copy()
mask_pixel = mask_pixel_energy.copy() if USE_MASK_PIXEL else np.zeros_like(holograms)
supportmask = supptemp

if holograms.shape[0] != n_energy:
    raise ValueError("The selected hologram stack does not match the energy axis")
if mask_pixel.shape != holograms.shape:
    raise ValueError("mask_pixel must have the same shape as holograms")

print("multi-energy hologram stack:", holograms.shape)
print("multi-energy mask stack:", mask_pixel.shape, "use mask:", USE_MASK_PIXEL)
print("support template:", supportmask.shape)
print("first energies:", energies_eV[:4])


In [ ]:
%matplotlib widget
plt.close("all")

show_indices = np.unique(np.linspace(0, n_energy - 1, 5, dtype=int))
fig, axes = plt.subplots(3, len(show_indices), figsize=(len(show_indices) * 3, 7), sharex=True, sharey=True)
for column, energy_index in enumerate(show_indices):
    axes[0, column].imshow(
        np.fft.fftshift(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(holograms[energy_index]))))),
        vmin=0,
        vmax=0.1,
    )
    axes[0, column].set_title(f"{SELECTED_POLARIZATION}, {energies_eV[energy_index]:.1f} eV")
    axes[0, column].axis("off")
    axes[1, column].imshow(np.log10(holograms[energy_index]))
    axes[1, column].set_title("Prepared hologram")
    axes[1, column].axis("off")
    axes[2, column].imshow(mask_pixel[energy_index])
    axes[2, column].set_title("Prepared mask_pixel")
    axes[2, column].axis("off")
plt.suptitle("Prepared single-polarization holograms, support, and mask")
plt.tight_layout()


## Multi-energy phase retrieval

## Same-energy opposite-helicity pair reconstructions

Before coupling a single-polarization stack across energy, reconstruct matching CR/CL hologram pairs independently at the same photon energies.


In [ ]:
offx,offy=0,0#-120,-220
import scipy
supptemp=np.zeros(holograms[0].shape)
from skimage.draw import disk
yy,xx=disk(tuple(np.array(supptemp.shape)/2+np.array([+offx,+offy])), 8)
supptemp[yy,xx]=1
#yy,xx=disk(((415+294)/2,(415+294)/2 ), 4+(415-294)/2)
#supptemp[yy,xx]=1
yy,xx=disk(((415+294)/2+offx,(415+294)/2+offy ), (+2+(415-294)/2))
supptemp[yy,xx]=1
yy,xx=disk((197+offx,512+offy), 8)
supptemp[yy,xx]=1

#supptemp*=(1.*(np.abs(interactive.reconstruct(holograms[0]))>9.e-6))


#supptemp*=(1.*(np.abs(interactive.reconstruct(holograms[0]))>0.5e-6))
supptemp = 1.*scipy.ndimage.binary_fill_holes(supptemp.astype(int))
mmask=np.zeros(supptemp.shape)
yy,xx=disk(tuple(np.array(supptemp.shape)/2), 10)
mmask[yy,xx]=1
interactive.cimshow((1+supptemp)*(np.log10(np.abs(interactive.reconstruct(holograms[4])))))
interactive.cimshow(mmask)

print(supptemp.shape)

In [ ]:
show_indices=np.array([0,7])

PAIR_RETRIEVAL_QUICK_RUN = QUICK_RUN if "QUICK_RUN" in globals() else False
PAIR_RETRIEVAL_INDICES = show_indices  # Use np.arange(n_energy) to run every energy pair.

if PAIR_RETRIEVAL_QUICK_RUN:
    pair_number_iterations = [5, 1, 5, 1]
    pair_average_img = [1, 1, 1, 1]
    pair_plot_every = [1e9, 1e9, 1e9, 1e9]
else:
    pair_number_iterations = [700, 50, 700, 50]
    pair_average_img = [30, 30, 30, 30]
    pair_plot_every = [350, 25, 350, 25]

pair_recipe = {
    "algorithm_list": ["HAPRE", "ER", "HAPRE", "ER"],
    "number_iterations": pair_number_iterations,
    "helicity": ["pos", "pos", "neg", "neg"],
    "beta_zero": [0.5, 0.5, 0.5, 0.5],
    "beta_mode": ["arctan", "const", "arctan", "const"],
    "alpha_zero": [0.4, 0., 0.4, 0.],
    "alpha_mode": ["arctan", "const", "arctan", "const"],
    "RL_its": [0, 0, 0, 0],
    "RL_freqs": [1e9, 1e9, 1e9, 1e9],
    "TV_freqs": [5, 1e9, 5, 1e9],
    "plot_every": pair_plot_every,
    "average_img": pair_average_img,
    "Fourier_last": [True, True, True, True],
    "hologram_intensity_cutoff_vmin": -1,
    "Startimage": [None, "pos", "pos", "neg"],
    "Startgamma": [None, None, None, None],
}
import scipy

same_energy_pair_results = {}
for energy_index in PAIR_RETRIEVAL_INDICES[:]:
    energy_index = int(energy_index)
    pair_mask = mask_pixel_energy[energy_index] if USE_MASK_PIXEL else np.zeros_like(cr_prepared[energy_index])
    results = pr.phase_retrieval_algorithm(
        cr_prepared[energy_index]*(1e-3*mmask+(1-mmask)),
        cl_prepared[energy_index]*(1e-3*mmask+(1-mmask)),
        np.clip(pair_mask+mmask,0,1),
        supptemp,
        phase_retrieval_recipe=pair_recipe,
    )
    same_energy_pair_results[float(energies_eV[energy_index])] = results
    print(f"retrieved CR/CL pair at {energies_eV[energy_index]:.1f} eV")

print(f"same-energy pair reconstructions: {len(same_energy_pair_results)}")


In [ ]:
fig,ax=plt.subplots(3,PAIR_RETRIEVAL_INDICES.size, figsize=(2*PAIR_RETRIEVAL_INDICES.size,6))

for ii,energy_index in enumerate(PAIR_RETRIEVAL_INDICES[:]):
    print(float(energies_eV[energy_index]))
    energy_index = int(energy_index)
    temp_p=same_energy_pair_results[float(energies_eV[energy_index])][0]
    temp_n=same_energy_pair_results[float(energies_eV[energy_index])][1]
    temp_p_pc=same_energy_pair_results[float(energies_eV[energy_index])][2]
    temp_n_pc=same_energy_pair_results[float(energies_eV[energy_index])][3]

    temp=(interactive.reconstruct(temp_p)-0*interactive.reconstruct(temp_n))

    roi=np.s_[294:415,294:415]

    
    ax[0,ii].imshow(np.real(temp[::-1,::-1])[roi])
    ax[1,ii].imshow(np.imag(temp[::-1,::-1])[roi])
    ax[2,ii].imshow(np.abs(temp[::-1,::-1])[roi])

    ax[0,ii].axis("off")
    ax[1,ii].axis("off")
    ax[2,ii].axis("off")

In [ ]:
plt.close("all")

interactive.cimshow(np.abs(temp_p)*mmask)

In [ ]:
QUICK_RUN = False

if QUICK_RUN:
    inner_Nit = [5, 1]
    outer_iterations = 2
    warmup_Nit = [20, 5]
else:
    inner_Nit = [50]
    outer_iterations = 12
    warmup_Nit = [300, 50]

recipe = {
    # Per-energy update schedule repeated during every outer iteration.
    "inner_mode": ["ER"],
    "inner_Nit": inner_Nit,
    "outer_iterations": outer_iterations,
    "warmup_mode": ["HAPRE", "ER"],
    "warmup_Nit": warmup_Nit,
    "shuffle_energies": True,
    "random_seed": 7,
    # Scalars are broadcast to all stages; lists customize each stage.
    "beta_zero": 0.5,
    "beta_mode": "arctan",
    "alpha_zero": 0.2,
    "alpha_mode": "linear_to_0",
    "TV_freq": 1e9,
    "RL_it": 0,
    "RL_freq": 1e9,
    "warmup_beta_zero": None,
    "warmup_beta_mode": None,
    "warmup_alpha_zero": None,
    "warmup_alpha_mode": None,
    "warmup_TV_freq": None,
    "warmup_RL_it": None,
    "warmup_RL_freq": None,
    "plot_every": 20,
    "average_img": 1,
    "Fourier_last": True,
    "final_fourier_constraint": False,
    "hologram_intensity_cutoff_vmin": -1,
    # Cross-energy object projection.
    "projection_model": "rank1_spectral",
    "rank": 1,
    "projection_every": 1,
    "projection_start": None,
    "projection_relaxation": 1.0,
    "projection_constraints_inside_support_only": False,
    "projection_static_mode": "mean",
    "energy_weights": None,
    "log_floor": 1e-12,
    # Rank-one spectral constraints; used only by rank1_spectral.
    "spectral_constraint": "known_beta",
    "energy_values": energies_eV,
    "known_beta_spectrum": np.imag(np.sum(effective_refractive_indices_m[:, -2:, 0], axis=1)),
    "known_delta_spectrum": np.real(np.sum(effective_refractive_indices_m[:, -2:, 0], axis=1)),
    "absorption_part": "imag",
    "kk_sign": 1.0,
    "kk_subtract_baseline": True,
    "kk_normalize_input": False,
    "known_beta_normalization": "none",
    "fit_known_beta_scale": True,
    "fit_known_beta_offset": True,
}

#supptemp=1.*scipy.ndimage.binary_dilation(supportmask, iterations=4)
import scipy
fields, fieldswarmup, components, bsmasks, errors = pr.multi_energy_phase_retrieval_algorithm(
    holograms,
    mask_pixel*0,
    supptemp,
    multi_energy_recipe=recipe,
)

print("projection model:", components.get("projection_model"))
print("final Fourier constraint applied:", components.get("final_fourier_constraint_applied"))
print(f"runtime: {errors['runtime_seconds']:.1f} s")


In [ ]:
holograms.shape, supptemp.shape

In [ ]:
plt.close("all")
interactive.cimshow(np.abs(interactive.reconstruct(holograms[4])))

In [ ]:
fig,ax=plt.subplots()
ax.plot(np.imag(np.sum(effective_refractive_indices_m[:, -2:, 0], axis=1)))

In [ ]:
%matplotlib widget
plt.close("all")


roi=np.s_[600:730,600:730]#200:330,200:330]
#roi=np.s_[444:580,444:580]

exit_waves = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(fieldswarmup, axes=(-2, -1)), axes=(-2, -1)), axes=(-2, -1))
show_indices = np.unique(np.linspace(0, n_energy - 1, 6, dtype=int))

fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices)*1.5 , 2*1.5), sharex=True, sharey=True)
for column, energy_index in enumerate(show_indices):
    mi,ma=np.percentile(np.abs(exit_waves[energy_index][roi])[supptemp[roi]==0] , (1,99))

    axes[0, column].imshow(np.abs(exit_waves[energy_index][roi]), vmin=mi, vmax=ma)
    axes[0, column].set_title(f"abs, {energies_eV[energy_index]:.1f} eV")
    axes[0, column].axis("off")
    axes[1, column].imshow(np.angle(exit_waves[energy_index][roi]), cmap="twilight")
    axes[1, column].set_title("phase")
    axes[1, column].axis("off")
plt.suptitle(f"Recovered exit waves ({SELECTED_POLARIZATION})")
plt.tight_layout()


In [ ]:
%matplotlib inline
plt.close("all")

exit_waves = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(fields, axes=(-2, -1)), axes=(-2, -1)), axes=(-2, -1))
show_indices = np.unique(np.linspace(0, n_energy - 1, 6, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices) * 3, 6), sharex=True, sharey=True)
for column, energy_index in enumerate(show_indices):
    axes[0, column].imshow(np.abs(exit_waves[energy_index][roi]))
    axes[0, column].set_title(f"abs, {energies_eV[energy_index]:.1f} eV")
    axes[0, column].axis("off")
    axes[1, column].imshow(np.angle(exit_waves[energy_index][roi]), cmap="twilight")
    axes[1, column].set_title("phase")
    axes[1, column].axis("off")
plt.suptitle(f"Recovered exit waves ({SELECTED_POLARIZATION})")
plt.tight_layout()


In [ ]:
components.keys()

In [ ]:
plt.close("all")
%matplotlib widget
interactive.cimshow((np.fft.fftshift(np.abs(components['spectral_spatial_map']))*supptemp)[280:430,280:430])

In [ ]:
for item in components.keys():
    print(item)

In [ ]:
fig,ax=plt.subplots()
key="spectrum_constrained"
ax.plot(np.real(components[key]))
ax2=ax.twinx()
ax2.plot(np.imag(components[key]), c="orange")

In [ ]:
components['spectral_coefficients']